<a href="https://colab.research.google.com/github/OranDanon/AI_Interview_submmision/blob/master/PayPal_Home_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task Overview: LLM-Powered Table Description Pipeline



*   MAKE A COPY OF THIS NOTEBOOK BY CLICKING **File -> Make a Copy to Drive**. If you can't find the upper panel,
expand te arrow in the top right corner.

*   Your implementation should begin bellow "Main Class to implement" (class TableExplainer). You can add as many classes as you think is necessary.

*   Run the describe_methods method to generate the descriptions and call dump_results to download your descriptions into a local (your Downloads folder)
JSON file.


## Introduction

This task involves developing a scalable pipeline that utilizes LLMs to automatically generate descriptive metadata for database tables based on provided SQL logic. Your goal is to create an endpoint capable of receiving a list of table logics and producing structured descriptions for each table in a consistent format. These descriptions should reflect the table's purpose, column meanings, and any relevant details derived from the SQL logic and it's source tables.

## Objective

This Pipeline should represent a production engine capable of generating descriptions for SQL tables on demand as part of a deployment process. The descriptions should be accurate, concise, and avoid making assumptions or hallucinating information that isn't supported by the provided SQL code or source data.

## Key Points

1. **Content Focus**: The descriptions should focus on explaining the content and purpose of each table and its columns, not the SQL syntax or specific SQL operations.

2. **Source Table Integration**: When generating descriptions, the pipeline should utilize relevant data from source tables to provide a comprehensive explanation of the derived tables.

3. **Attention for Dependencies**: In some cases (like the case in the table logics you recieved) there could be some tables that are depending on other tables. In this case, it is advised to pay attention to the order of description so the description of the upstream table will be available to use when describing the downstream table.

4. **Production ready code**: The solution must be capable of handling thousands of derived tables efficiently.

5. **Accuracy Over Guesswork**: The pipeline should only describe what it knows.





## Data Structure Overview

### Input Data Structure

The input to the pipeline consists of a list of JSON objects, each representing a derived table. These JSON objects include:

- **name**: The name of the table.
- **DDL**: The Data Definition Language (DDL) statements that define the table’s structure, including columns and data types.
DDL commands are used to define, modify, and remove database structures. These commands do not manipulate data directly but rather define the blueprint of how data is stored in the database. The common DDL commands include: CREATE, ALTER,
DROP, TRUNCATE, RENAME.

- **DML**: The Data Manipulation Language (DML) is the group of commands that describe how data is manipulated within the table, such as inserting, updating, or selecting data, deleting rows.

- **sources**: A list of source tables referenced in the DDL and DML of the derived table.



### Output Data Structure

The output of the main function is a list of dictionaries, each providing a detailed description of a derived table. Each dictionary includes:

- **name**: The name of the table.
- **table_description**: A concise explanation of the table’s purpose and the type of data it holds.
- **descriptions**: A dictionary where each key is a column name and the value is a description of that column’s role and characteristics.

### Example Structure

**Input - table_logics (Example JSON Object):**
```json
{
    "name": "User_Loyalty",
    "DDL": "CREATE TABLE User_Loyalty (\n\tusr_id INT PRIMARY KEY,\n\tloyalty_status VARCHAR(255),\n\tpoints_summary TEXT,\n\trecent_activity TEXT\n);",
    "DML": "INSERT INTO User_Loyalty (usr_id, loyalty_status, points_summary, recent_activity)\nSELECT \n\tus.usr_id,\n\tCONCAT(us.seg_typ, '-', CASE WHEN us.seg_score > 60 THEN 'Gold' ELSE 'Silver' END) AS loyalty_status,\n\tCONCAT('Earned:', SUM(CASE WHEN ts.ttl_txns > 20 THEN ROUND(ts.ttl_amnt * 0.1, 2) ELSE 0 END), '|Redeemed:', SUM(CASE WHEN ts.ttl_amnt > 500 THEN ROUND(ts.ttl_amnt * 0.05, 2) ELSE 0 END)) AS points_summary,\n\tSTRING_AGG(CASE WHEN ts.ttl_amnt > 100 THEN CONCAT(ts.last_txn, ':HighSpending') ELSE CONCAT(ts.last_txn, ':Normal') END, '~') WITHIN GROUP (ORDER BY ts.last_txn DESC) AS recent_activity\nFROM \n\tUser_Segments us\nJOIN \n\tTransaction_Summary ts ON us.usr_id = ts.usr_id\nGROUP BY \n\tus.usr_id, us.seg_typ, us.seg_score;",
    "sources": ["User_Segments", "Transaction_Summary"]
}
```

**Output - table_descriptions (Example JSON Object):**
```json
{
    "name": "Merchant_Ratings",
    "table_description": "This table tracks ratings given to merchants, providing insights into their performance based on user feedback.",
    "descriptions": {
        "mrch_id": "Unique identifier for each merchant. Inherited from the Merchants table.",
        "rating_value": "The rating assigned to the merchant by users, on a scale of A to F.",
        "rating_date": "The date when the rating was recorded."
    }
}
```

### Additional Resources

Description dictionaries are available for the three base source tables (`Users`, `Transactions`, `Merchants`). These descriptions help establish the foundational logic and data structures within the organization, providing essential context for understanding the derived tables.

# Baseline Gemini Endpoint

In [ ]:
!pip install -q -U google-generativeai
!pip install retry


In [ ]:
# @markdown ## Enter your Gemini API Key
# @markdown [Click here to learn how to create your Gemini API token](https://ai.google.dev/gemini-api/docs/api-key)
API_KEY = "" # @param {type:"string"}

# Export the API key as an environment variable
import os
os.environ['API_KEY'] = API_KEY

# Confirm that the API key has been set
if API_KEY.strip() != "":
    print("API key has been set successfully!")
else:
    print("Please enter a valid API key.")


In [ ]:
# Gemini class definition
import os
from retry import retry

import google.generativeai as genai
from  google.api_core.exceptions import TooManyRequests, InternalServerError
from  google.generativeai.types.generation_types import GenerationConfig


class Gemini:
    def __init__(self):
        genai.configure(api_key=os.environ["API_KEY"])
        self.model = genai.GenerativeModel('gemini-1.5-flash')

    @retry((InternalServerError, TooManyRequests), tries=3, delay=90)
    def run_query(self, query, temperature=0, top_k=32, max_output_tokens=2048, top_p=1, candidate_count=None,
                  stop_sequences=None, response_mime_type=None, response_schema=None):

        config = GenerationConfig(max_output_tokens=max_output_tokens, temperature=temperature, top_p=top_p, top_k=top_k,
                                  candidate_count=candidate_count, stop_sequences=stop_sequences,
                                  response_mime_type=response_mime_type, response_schema=response_schema)
        response = self.model.generate_content(query, generation_config=config)
        return response.text

In [ ]:
# Usage example

gemini = Gemini()
print(gemini.run_query("hello, what is the meaning of the universe?"))

In [ ]:
# @markdown # Overview of the Uploaded File

# @markdown ## Run this cell to upload the zipped folder files from your local computer.
# @markdown 1. **source_descriptions:** This folder contains descriptions of the source tables. These descriptions are crucial for understanding the context of the data and should be used in your code.
# @markdown 2. **table_logics:** This is your input folder and it contains all the logic files for the tables that need to be explained.
# @markdown 3. **table_descriptions:** This folder includes an example of how the final results should look. **IMPORTANT!!** This example is for reference only and should not be used in your code or prompts. Also, this directory contains only a sample subset and doesn't include all the final tables.

from google.colab import files
import zipfile
import os

# Let the user upload the file
uploaded = files.upload()

# Extract the uploaded ZIP file
for file_name in uploaded.keys():
    # Ensure the uploaded file is a zip file
    if file_name.endswith(".zip"):
        output_dir = "/content"

        # Extract the contents of the zip file
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall(output_dir)

        print(f"Files extracted to {output_dir}")
    else:
        print("Please upload a zip file.")


# Main Class to implement

Implement the main describe function under the given class

In [ ]:
import json
import os


from google.colab import files

class TableExplainer:

  def __init__(self):
    self.model = Gemini()


  def dump_results(self, total_results: list[dict]):
    # Use this method at the end of your assignment to store all results
    # into a JSON file and download it automatically into the Downloads folder.
    file_path = os.path.join('/content/', 'pp_assignment_results.json')

    with open(file_path, 'w') as f:
          json.dump(total_results, f, indent=4)

    files.download(file_path)


  def describe_tables(self, table_logics: list[dict]) -> list[dict]:
    # this is the main function to implement.
    # The input 'table_logics' is a list of tables logics to be described (provided in the jsons)
    # The output is a list of description dicts, similar to the source descriptions or the reference descriptions files

    # input example:

    #[{
    # "name": "Country_Loyalty",
    # "DDL": "CREATE TABLE Country_Loyalty (\n\tctry VARCHAR(50) PRIMARY KEY,\n\ttotal_loyalty_score DECIMAL(15, 2),\n\taverage_loyalty_status DECIMAL(5, 2),\n\tactive_user_count INT\n);",
    # "DML": "INSERT INTO Country_Loyalty (ctry, total_loyalty_score, average_loyalty_status, active_user_count)\nSELECT \n\tcs.ctry,\n\tSUM(ul.seg_score) AS total_loyalty_score,\n\tROUND(AVG(CASE WHEN ul.loyalty_status LIKE '%Gold%' THEN 3 WHEN ul.loyalty_status LIKE '%Silver%' THEN 2 ELSE 1 END), 2) AS average_loyalty_status,\n\tCOUNT(DISTINCT us.usr_id) AS active_user_count\nFROM \n\tCountry_Statistics cs\nJOIN \n\tUser_Loyalty ul ON cs.ctry = ul.ctry\nJOIN \n\tUser_Segments us ON ul.usr_id = us.usr_id\nWHERE \n\tcs.ttl_usrs > 100 AND us.seg_score > 60\nGROUP BY \n\tcs.ctry;",
    # "sources": ["Country_Statistics", "User_Loyalty", "User_Segments"]
# }]


    # output example:

    # [{
    #"name": "Country_Loyalty",
    #"table_description": "This table aggregates user loyalty data by country, providing insights into total loyalty scores and active user counts.",
    #"descriptions": {
        # "ctry": "The country in which the users are registered. Inherited from the Users table.",
        # "total_loyalty_score": "The total loyalty score across all users in the country.",
        # "average_loyalty_status": "The average loyalty status of users in the country, derived from individual user statuses.",
        # "active_user_count": "The number of active users in the country with a loyalty score above a certain threshold."
    #}
#}]
    pass